In [8]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os
load_dotenv()

True

### Gettting all the components ready
> Document loader

> Splitter

> Embedding model

> Vector DB 

> LLM Model 

In [6]:
def doc_loader(path):
    try:
        loader = DirectoryLoader(path,
                                glob="**/*.pdf",
                                loader_cls=PyMuPDFLoader,
                                show_progress=True)
        documents = loader.load()
        print(f"Loaded {len(documents)} documents from {path}")
        return documents
    except Exception as e:
        print(f"Error loading documents from {path}: {e}")
        return None

def text_splitter(documents):
    print("Splitting documents into chunks...")
    try:
        if not documents:
            raise ValueError("No documents to split")
        splitter = RecursiveCharacterTextSplitter(chunk_size=1000, 
                                              chunk_overlap=150,
                                              length_function=len,
                                              separators=["\n\n", "\n", " ", ""])
        chunks = splitter.split_documents(documents)
        print(f"""Split into {len(chunks)} chunks
            Document splitting complete""")
        return chunks
    except Exception as e:
        print(f"Error splitting documents: {e}")
        return None

def create_vector(chunks):
    print('Loading embedding model...')
    embedding = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
        )
    print('Creating vector store...')
    vector_store = Chroma.from_documents(chunks, 
                                         embedding, 
                                         collection_name="pdf_docs")
    print('Vector store created successfully')
    return vector_store

def llm_model():
    print('Loading LLM model...')
    llm = ChatOpenAI(
        model="moonshotai/kimi-k2.6:free", 
        api_key=os.getenv("OPENROUTER_API_KEY"),
        base_url="https://openrouter.ai/api/v1")
    return llm

### Assembeling the components

In [9]:
print('LLM Checking the query if any query reformulation is needed...')
def rewrite_query(llm,query):
    llm = llm_model()
    prompt = f"""
        Rewrite the question only if necessary for document retrieval.
        Preserve the original meaning.
        Return only the rewritten query.
        Question: {query}
"""

    rewritten_query = llm.invoke(prompt)
    return rewritten_query.content

print('Starting generation process...')
def generation():
    llm = llm_model()
    documents = doc_loader("../data")
    chunks = text_splitter(documents)
    vector_store = create_vector(chunks)
    retriever = vector_store.as_retriever(
        search_kwargs={"k": 3}
        )
    
    user_query = input("Enter your query: ")
    rewritten_query = rewrite_query(llm,user_query)
    print(f"Rewritten query: {rewritten_query}")
    results = retriever.invoke(rewritten_query)

    context = "\n".join(doc.page_content for doc in results)
    final_prompt = PromptTemplate.from_template(
        """
        Use the following context to answer the question: 
        Context : {context}
        Question: {query}
        """
    )
    final_response = llm.invoke(
        final_prompt.format(
            context=context, 
            query=rewritten_query
            )
        )
    
    print(f"Final response: {final_response.content}")
if __name__ == "__main__":
    generation()

LLM Checking the query if any query reformulation is needed...
Starting generation process...
Loading LLM model...


100%|██████████| 3/3 [00:00<00:00,  3.13it/s]


Loaded 781 documents from ../data
Splitting documents into chunks...
Split into 1573 chunks
            Document splitting complete
Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5393.22it/s]


Creating vector store...
Vector store created successfully
Loading LLM model...
Rewritten query: Who is Sung Jinwoo?
Final response: According to the context, Sung Jinwoo is **the S-rank hunter who vanished without a trace after completely ruining Minsung Lee’s press conference**.
